In [1]:
import pandas as pd

In [2]:
df=pd.read_csv("Titanic_Dataset.csv")
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708


In [3]:
df.duplicated().sum()

np.int64(0)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
dtypes: float64(2), int64(5), str(3)
memory usage: 69.7 KB


In [5]:
df = df.drop(columns=["PassengerId", "Name", "Ticket","Survived"])

In [6]:
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

In [7]:
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
trf1=ColumnTransformer([(
    "impute_numerical",
    SimpleImputer(strategy='mean'),
    ['Age']
)],
remainder='passthrough',
verbose_feature_names_out=False).set_output(transform='pandas')

In [9]:
from sklearn.preprocessing import OneHotEncoder
trf2=ColumnTransformer([(
    "OHE",
    OneHotEncoder(sparse_output=False),
    ['Sex','Pclass']
    )],
    remainder='passthrough',
    verbose_feature_names_out=False).set_output(transform='pandas')

In [10]:
from sklearn.preprocessing import MinMaxScaler

trf3=ColumnTransformer([(
    'scaling',
    MinMaxScaler(),
    ["Age", "SibSp", "Parch","Fare","FamilySize"]
)],
remainder='passthrough',
verbose_feature_names_out=False).set_output(transform='pandas')

In [11]:
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ('trans1',trf1),
    ('trans2',trf2),
    ('trans3',trf3)

])

In [12]:
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('trans1', ...), ('trans2', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('impute_numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"verbose_feature_names_out verbose_feature_names_out: 

In [13]:
df_transformed=pipe.fit_transform(df)

In [14]:
df_transformed

,Age,SibSp,Parch,Fare,FamilySize,Sex_female,Sex_male,Pclass_1,Pclass_2,Pclass_3,IsAlone
0,0.271174,0.125,0.000000,0.014151,0.1,0.0,1.0,0.0,0.0,1.0,0
1,0.472229,0.125,0.000000,0.139136,0.1,1.0,0.0,1.0,0.0,0.0,0
2,0.321438,0.000,0.000000,0.015469,0.0,1.0,0.0,0.0,0.0,1.0,1
3,0.434531,0.125,0.000000,0.103644,0.1,1.0,0.0,1.0,0.0,0.0,0
4,0.434531,0.000,0.000000,0.015713,0.0,0.0,1.0,0.0,0.0,1.0,1
...,...,...,...,...,...,...,...,...,...,...,...
886,0.334004,0.000,0.000000,0.025374,0.0,0.0,1.0,0.0,1.0,0.0,1
887,0.233476,0.000,0.000000,0.058556,0.0,1.0,0.0,1.0,0.0,0.0,1
888,0.367921,0.125,0.333333,0.045771,0.3,1.0,0.0,0.0,0.0,1.0,0
889,0.321438,0.000,0.000000,0.058556,0.0,0.0,1.0,1.0,0.0,0.0,1
